In [1]:

import importlib.util

for package in ["xgboost", "lightgbm"]:
    installed = importlib.util.find_spec(package) is not None
    print(f"{package}: {'Installed' if installed else 'Not installed'}")


xgboost: Installed
lightgbm: Installed


In [ ]:

import numpy as np
import pandas as pd

from pathlib import Path
import sys
from scipy.stats import loguniform, randint, uniform

from sklearn.svm import SVR
from sklearn.ensemble import (
    GradientBoostingRegressor,
    VotingRegressor
)

from sklearn.compose import TransformedTargetRegressor
from sklearn.model_selection import RandomizedSearchCV, train_test_split, KFold
from sklearn.metrics import (
    mean_squared_log_error,
    mean_absolute_error,
    mean_squared_error,
    make_scorer
)

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

sys.path.append(str(Path().resolve().parent))
from src.preprocessing import build_pipeline
from src.feature_engineering import add_engineered_features


In [6]:

DATA_DIR = Path("../data")

if not (DATA_DIR / "train.csv").exists():
    DATA_DIR = Path("data")

train_df = pd.read_csv(DATA_DIR / "train.csv")

X = train_df.drop(columns=["Id", "SalePrice"])
y = train_df["SalePrice"]

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("X_train:", X_train.shape)
print("X_valid:", X_valid.shape)


X_train: (1168, 79)
X_valid: (292, 79)


In [7]:

X_train_fe = add_engineered_features(
    X_train,
    include_interactions=True
)

X_valid_fe = add_engineered_features(
    X_valid,
    include_interactions=True
)

print("Training:", X_train_fe.shape)
print("Validation:", X_valid_fe.shape)

assert list(X_train_fe.columns) == list(X_valid_fe.columns)


Training: (1168, 86)
Validation: (292, 86)


In [ ]:
cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)